In [ ]:
import pandas as pd
from argopy import DataFetcher
import xarray as xr

argo_loader = DataFetcher()
ds = argo_loader.region([32, 80, 8, 26, 0, 2000, '2023-03-01', '2023-03-31']).to_xarray()
ds = ds.to_dataframe().reset_index()
columns_to_keep = [
    'PLATFORM_NUMBER', 
    'CYCLE_NUMBER', 
    'LATITUDE', 
    'LONGITUDE', 
    'TIME',
    'PRES', 
    'TEMP', 
    'PSAL'
]
ds.dropna(inplace=True)
ds = ds[columns_to_keep]
ds.head()

/home/rohith/Projects/Poseidon-RAG/Backend/.venv/bin/python: No module named pip


,PLATFORM_NUMBER,CYCLE_NUMBER,LATITUDE,LONGITUDE,TIME,PRES,TEMP,PSAL
0,2903142,12,12.98423,59.08548,2023-03-01 13:50:59,1.12,26.587000,36.400200
1,2903142,12,12.98423,59.08548,2023-03-01 13:50:59,2.04,26.587999,36.407001
2,2903142,12,12.98423,59.08548,2023-03-01 13:50:59,2.96,26.589001,36.405701
3,2903142,12,12.98423,59.08548,2023-03-01 13:50:59,3.96,26.584999,36.402401
4,2903142,12,12.98423,59.08548,2023-03-01 13:50:59,5.00,26.558001,36.410400


In [2]:
ds.info()

<class 'pandas.DataFrame'>
Index: 89679 entries, 0 to 94699
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   PLATFORM_NUMBER  89679 non-null  int64         
 1   CYCLE_NUMBER     89679 non-null  int64         
 2   LATITUDE         89679 non-null  float64       
 3   LONGITUDE        89679 non-null  float64       
 4   TIME             89679 non-null  datetime64[ns]
 5   PRES             89679 non-null  float32       
 6   TEMP             89679 non-null  float32       
 7   PSAL             89679 non-null  float32       
dtypes: datetime64[ns](1), float32(3), float64(2), int64(2)
memory usage: 5.1 MB


In [3]:
import os
import sqlite3

conn = sqlite3.connect(os.path.join(os.getcwd(),'data','argo_data.db'))
ds.to_sql('measurements',con=conn,index=False,if_exists='replace')

89679

In [4]:
query = '''
    SELECT 
        Platform_number,
        COUNT(*) as total_readings,
        ROUND(MIN(temp), 2) as min_temp,
        ROUND(MAX(temp), 2) as max_temp,
        ROUND(AVG(latitude), 4) as avg_lat,
        ROUND(AVG(longitude), 4) as avg_lon
    FROM measurements
    GROUP BY Platform_number
'''

query ="""
        select Count(distinct(Platform_number)) from measurements
        """

cursor = conn.cursor()
cursor.execute(query)

rows = cursor.fetchall()

print(rows)

[(19,)]
